# NB0: Controlled Sandbox Preflight

**Purpose:** Before learners touch the agent team, verify the classroom environment. This notebook proves the core path is offline, deterministic, writable, and ready for Jupyter execution.

**Course manager rule:** Live LLM execution is intentionally disabled by default. Learners should be able to complete the governance shell without API keys, network access, or paid model calls.

In [ ]:
import importlib
import os
import pathlib
import tempfile

REQUIRED_PACKAGES = ["pydantic", "pytest", "fastapi"]
OPTIONAL_PACKAGES = ["httpx", "jupyter", "nbval", "crewai", "chromadb", "instructor", "openai"]

print("=== Required Package Check ===")
for package in REQUIRED_PACKAGES:
    module = importlib.import_module(package)
    version = getattr(module, "__version__", "installed")
    print(f"OK: {package} ({version})")

print("\n=== Optional Package Check ===")
for package in OPTIONAL_PACKAGES:
    spec = importlib.util.find_spec(package)
    if spec is None:
        print(f"optional missing: {package}")
    else:
        # Do not import optional frameworks here. Some packages perform
        # startup writes outside the sandbox when imported.
        print(f"optional available: {package}")

In [ ]:
print("=== Offline Deterministic Mode ===")
USE_LIVE_LLM = False
assert USE_LIVE_LLM is False

live_keys_present = {
    name: bool(os.environ.get(name))
    for name in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY"]
}
print("Live keys present but not required:", live_keys_present)
print("Network access required for core notebooks: False")

In [ ]:
print("=== Working Directory Check ===")
cwd = pathlib.Path.cwd()
print("cwd:", cwd)

expected_markers = ["notebooks", "assignments", "src", "tests"]
found = {marker: (cwd / marker).exists() for marker in expected_markers}
print("repo markers:", found)
assert all(found.values()), "Run notebooks from the repository root."

In [ ]:
print("=== /tmp Write Check ===")
tmp_dir = pathlib.Path("/tmp") if pathlib.Path("/tmp").exists() else pathlib.Path(tempfile.gettempdir())
probe = tmp_dir / "managing_multi_agent_teams_preflight.txt"
probe.write_text("sandbox write ok\n")
assert probe.read_text() == "sandbox write ok\n"
probe.unlink()
print(f"OK: wrote and cleaned test file in {tmp_dir}")

## Controlled Sandbox Inputs

- Execution mode: `offline_deterministic`
- Live LLM flag: `USE_LIVE_LLM = False`
- Network access: disabled for core learning path
- Required package tier: `pydantic`, `pytest`, `fastapi`
- Optional package tier: `httpx`, `jupyter`, `nbval`, `crewai`, `chromadb`, `instructor`, `openai`
- Output policy: clear notebook outputs before commit; instructor copies may keep rendered outputs
- Alias policy: keep compatibility aliases, expose canonical NB0-NB11 to learners

**The Takeaway:** The learning environment is a governed sandbox too. Before managing agents, verify the room where the agents run.